# The Anatomy of AI · Lecture 31 · Capstone: One Neuron, Completely
### The notebook

The course ends with a method: fifteen steps for explaining one neuron *completely*, and a neuron to try them on.
Goodfire's essay left one InceptionV1 neuron unfinished, **5a:719**. In its words, it "collects a pinched-center shape:
bow ties, butterflies, swimsuit tops. But it also picks up things that go with them, men's dress shoes with the bow ties
and beach bags and flip-flops with the swimsuit tops … I don't fully understand these exceptions yet."

Here you take 5a:719 through the steps the lecture did, with your own hands: its number, its spread, both poles, its
wiring, a drawing, a controlled test, an explanation that can lose, and what happens when you take it away. Every result
the lecture states is checked with `check(...)`. Where we use a small sample of photos, the check compares with the
lecture's own scan of the same photos.

Five **Your turn** exercises along the way; worked solutions at the end. The last section is the assignment.

It runs on free Colab in about 15 minutes on a **T4 GPU**. It downloads InceptionV1 (27 MB) and about 3,400 small photos.
No keys.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "scipy==1.14.1", "pillow==11.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"       # the page shows results, not download bars
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{'' if tol < 1e-6 or not isinstance(expected, (int, float)) else f' ± {tol:g}'})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

In [ ]:
# ImageNet's 50,000 validation photos, one at a time. The course used a public 256-pixel copy
# (benjamin-paine/imagenet-1k-256x256 on Hugging Face). Hugging Face's dataset viewer hands out any row by its number,
# so we fetch only the photos we need. Photo n here is photo n in the lecture.
import requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

ROWS = ("https://datasets-server.huggingface.co/rows?dataset=benjamin-paine/imagenet-1k-256x256"
        "&config=default&split=validation&offset={}&length={}")
os.makedirs("photos", exist_ok=True)


def _get(url, tries=10):
    """GET with patience: the viewer limits how fast one machine may ask, so back off when it says so (429)."""
    status = None
    for k in range(tries):
        try:
            r = requests.get(url, timeout=60)
            status = r.status_code
            if status == 200:
                return r
        except requests.RequestException as e:
            status = type(e).__name__
        time.sleep(min(60, 2 ** k) + np.random.rand())
    raise RuntimeError(f"could not fetch {url[:90]}... (last status {status})")


def _fetch_block(start, n):
    """Rows start..start+n-1 (at most 100) -> {row: (label, jpeg bytes)}, cached on disk."""
    need = [i for i in range(start, start + n) if not os.path.exists(f"photos/{i}.jpg")]
    if need:
        rows = _get(ROWS.format(start, n)).json()["rows"]
        def one(row):
            i = row["row_idx"]
            open(f"photos/{i}.jpg", "wb").write(_get(row["row"]["image"]["src"]).content)
            open(f"photos/{i}.label", "w").write(str(row["row"]["label"]))
        with ThreadPoolExecutor(8) as ex:
            list(ex.map(one, [r for r in rows if r["row_idx"] in need]))


def fetch(ids):
    """Download the photos with these numbers (and cache them)."""
    ids = sorted(set(int(i) for i in ids))
    todo = [i for i in ids if not os.path.exists(f"photos/{i}.jpg")]
    blocks = sorted({(i, 1) for i in todo})
    with ThreadPoolExecutor(3) as ex:              # a few row requests at a time
        list(ex.map(lambda b: _fetch_block(*b), blocks))


def fetch_range(start, n):
    """Download a contiguous run of photos, 100 rows per request."""
    for s in range(start, start + n, 100):          # one block of 100 rows at a time
        _fetch_block(s, min(100, start + n - s))


def photo(i, size=224):
    """Photo i as InceptionV1 saw it: 256 x 256, centre crop 224 x 224, RGB uint8."""
    im = Image.open(f"photos/{int(i)}.jpg").convert("RGB")
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    o = (256 - size) // 2
    return np.asarray(im.crop((o, o, o + size, o + size)), dtype=np.uint8)


def label(i):
    return int(open(f"photos/{int(i)}.label").read())

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network the essay and this lecture study, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()
assert "k=2" in _f.read_text(), "the normalization fix did not take"

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):          # torch.hub's download bar; one line below says it instead
    net = inceptionv1(pretrained=True).eval().to(DEV)
for p in net.parameters():
    p.requires_grad_(False)
print(f"InceptionV1 loaded ({sum(p.numel() for p in net.parameters()) / 1e6:.1f} million weights, a 27 MB download)")

BRANCHES = ["1x1", "3x3", "5x5", "pool_reduce"]


class Taps:
    """Record each layer's value *before* its ReLU, the four branches side by side in channel order,
    so 4d:295 is channel 295 of mixed4d. Before the ReLU both poles of a neuron are visible."""
    def __init__(self, layers):
        self.parts = {}
        for L in layers:
            for b in BRANCHES:
                getattr(net, f"{L}_{b}_pre_relu_conv").register_forward_hook(self._keep(L, b))

    def _keep(self, L, b):
        def f(_m, _i, out):
            self.parts[(L, b)] = out
        return f

    def __getitem__(self, L):
        return torch.cat([self.parts[(L, b)] for b in BRANCHES], 1)


def to_input(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


def batches(imgs, size=64):
    for s in range(0, len(imgs), size):
        x = torch.from_numpy(np.stack(imgs[s:s + size])).to(DEV).permute(0, 3, 1, 2).float() / 255
        yield to_input(x)

In [ ]:
NAMES = requests.get("https://datasets-server.huggingface.co/info?dataset=benjamin-paine/imagenet-1k-256x256",
                     timeout=60).json()["dataset_info"]["default"]["features"]["label"]["names"]
NAMES = [n.split(",")[0].strip() for n in NAMES[:1000]]
LUCID = requests.get("https://storage.googleapis.com/modelzoo/labels/ImageNet_alternate.txt", timeout=60).text.splitlines()
taps = Taps(["mixed4e", "mixed5a", "mixed5b"])
L5, N = "mixed5a", 719

## Plate I · What "completely" means

The fifteen steps (the lecture's checklist; those marked * are required):

| # | step | what it means |
|---|---|---|
| 1* | Identify the neuron | which number is its activation |
| 2* | Its distribution | the histogram of its peaks; the strong half |
| 3* | Positive pole | what drives it highest, and the class mix |
| 4* | Negative pole | the same, most negative |
| 5* | Describe it without nouns | one sentence per pole |
| 6* | Wiring in | its strongest inputs |
| 7* | Wiring out | its strongest readers |
| 8 | Weak activations | strong, middle and weak bands |
| 9 | Feature visualization | a drawing; does it match the photos? |
| 10* | A controlled test, written first | vary one property, report the failures |
| 11* | An explanation that can lose | predicted, near misses, distant |
| 12* | Take it away | ablate; which outputs lose most |
| 13 | Growth | over training (InceptionV1 has no checkpoints) |
| 14* | Alternatives and residue | what is left unexplained |
| 15* | The explanation | one sentence, its scope, your confidence |

## Plate IV · Steps 1, 2 and 8: what number, and how spread

**Step 1.** 5a:719 is channel 719 of layer 5a, read *before* its ReLU. At each of 7 × 7 positions it has a value; a
photo's **peak** is the largest. 5a's channels run over four branches, and 719 is in the last, the **pool branch**:
channel 15 of 128.

**Step 2.** Its spread over photos. The lecture measured all 50,000: it fires (peak above 0) on 69% of photos. Here we
take photos 0–2,999 and compare with the lecture's scan of the same slice.

In [ ]:
sizes = [getattr(net, f"{L5}_{b}_pre_relu_conv").out_channels for b in BRANCHES]
print("5a's branches:", dict(zip(BRANCHES, sizes)))
off = sum(sizes[:3])
check("5a:719's branch", BRANCHES[3] if off <= N < off + sizes[3] else None, "pool_reduce")
check("its index inside the branch", N - off, 15)

fetch_range(0, 3000)
S = list(range(3000))


def peaks(ids, layer=L5, ch=N):
    fetch(ids)
    mx, mn = [], []
    with torch.no_grad():
        for x in batches([photo(i) for i in ids]):
            net(x)
            a = taps[layer][:, ch].flatten(1)
            mx += a.amax(1).tolist(); mn += a.amin(1).tolist()
    return np.array(mx), np.array(mn)


mx, mn = peaks(S)
labels = np.array([label(i) for i in S])
check("share of the slice where it fires (peak > 0)", (mx > 0).mean(), 0.6920, tol=0.01)
check("median peak on the slice", float(np.median(mx)), 20.023, tol=1.5)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(mx, bins=60, color=PAPER, edgecolor=INK)
ax.axvline(0, color=INK2, lw=1); ax.axvline(0.5 * mx.max(), color=INK, ls="--", lw=1)
ax.text(0.5 * mx.max(), ax.get_ylim()[1] * 0.9, "  half its strongest", fontsize=9)
ax.set_xlabel("5a:719's peak on a photo"); ax.set_ylabel("photos"); plt.tight_layout(); plt.show()
print(f"strong half (peak >= half the slice's max): {(mx >= 0.5 * mx.max()).sum()} of {len(mx)} photos")

Most photos sit in a broad middle; a thin tail fires strongly. Step 8 asks what each band means. The lecture's habit: the
neuron's *job* is read from the strong tail, and the middle is checked separately rather than assumed.

## Plate V · Steps 3–4: both poles

In [ ]:
top = np.argsort(-mx)[:30]
LECTURE_TOP30 = [893, 615, 490, 679, 545, 763, 636, 777, 748, 721, 496, 457, 777, 414, 774, 481, 902, 723, 695, 723, 266, 546, 630, 910, 464, 427, 684, 502, 556, 921]     # the lecture's scan: the classes of this slice's 30 strongest photos
same = len(set(np.array(S)[top]) & set([865, 685, 2833, 849, 2760, 2839, 1637, 38, 1564, 437, 1907, 201, 2305, 2182, 2425, 1673, 1244, 2733, 2907, 990, 1233, 1835, 2897, 2684, 2547, 1882, 1682, 2039, 1412, 979]))
check("of the slice's 30 strongest photos, the same photos as the lecture's", same, 30, tol=3)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for a, j in zip(axes[0], top[:8]):
    a.imshow(photo(S[j])); a.axis("off"); a.set_title(NAMES[labels[j]][:14], fontsize=8)
for a, j in zip(axes[1], np.argsort(mn)[:8]):
    a.imshow(photo(S[j])); a.axis("off"); a.set_title(NAMES[labels[j]][:14], fontsize=8)
axes[0, 0].text(-40, 112, "strongest", rotation=90, va="center"); axes[1, 0].text(-40, 112, "weakest", rotation=90, va="center")
plt.tight_layout(); plt.show()
from collections import Counter
print("classes of the 30 strongest:", Counter(NAMES[c] for c in labels[top]).most_common(8))

## Plate II · The unfinished neuron, and Step 11: an explanation that can lose

Before measuring anything, the lecture wrote down, from the essay's description alone, which classes 5a:719 should
answer and which it should not:

- **predicted** (a pinched centre): bow tie, six butterflies, bikini, maillot, brassiere, hourglass, dumbbell;
- **near misses** (tie- or insect-like, no pinch): Windsor tie, dragonfly, damselfly, lacewing, swimming trunks;
- **residue**, the essay's "exceptions" that have ImageNet classes: Loafer, sandal;
- **distant**: pizza, volcano, goldfish, valley, traffic light.

A photo is a **hit** if its peak reaches 30% of the neuron's strongest value in all of ImageNet-val (90.4). The
lecture, with all 50 photos per class: the predicted classes hit **26%**, the near misses **15%**, the residue
**71%**, the distant **1%**. The description loses its own test: the "exceptions" fire far more than the pinched
shapes. Here, the first 10 photos of each class.

In [ ]:
GROUPS = {"predicted": [["bow tie", [125, 201, 2037, 3044, 3906, 3933, 4838, 4942, 4986, 5568]], ["admiral", [1186, 1214, 3480, 4452, 4999, 6818, 7015, 8662, 8692, 9858]], ["ringlet", [809, 1014, 4589, 4849, 5785, 6513, 7197, 8141, 11073, 11910]], ["monarch", [1289, 1716, 2934, 5136, 5613, 5951, 6607, 7582, 11142, 11373]], ["cabbage butterfly", [30, 1013, 4010, 4074, 7367, 8544, 10024, 11174, 11588, 11779]], ["sulphur butterfly", [580, 617, 2378, 2905, 3497, 5116, 5903, 7945, 10430, 11160]], ["lycaenid", [1535, 2786, 5156, 9022, 10101, 11978, 13186, 13321, 13439, 17099]], ["bikini", [238, 1071, 4708, 5004, 5307, 5482, 6331, 6473, 7558, 9804]], ["maillot", [116, 183, 425, 1561, 2512, 3006, 3824, 4530, 6978, 7052]], ["brassiere", [807, 3066, 4598, 5422, 7483, 8377, 8442, 8953, 9815, 10548]], ["hourglass", [1942, 3950, 3985, 4081, 4728, 6118, 6240, 7212, 7276, 7715]], ["dumbbell", [1044, 1274, 1286, 4024, 4948, 5500, 6778, 7619, 7948, 10054]]], "near_miss": [["Windsor tie", [105, 284, 1180, 1654, 4157, 4615, 4857, 5425, 5651, 5795]], ["dragonfly", [417, 1237, 1751, 2224, 5157, 6759, 7424, 8000, 10149, 10924]], ["damselfly", [638, 964, 1841, 1920, 2581, 2892, 3238, 4214, 4259, 6312]], ["lacewing", [1465, 2026, 4660, 8068, 11012, 11600, 11782, 12044, 14030, 14963]], ["swimming trunks", [64, 1005, 1165, 1917, 2799, 3564, 5965, 8807, 11186, 15590]]], "residue": [["Loafer", [384, 2389, 2897, 3254, 5723, 6518, 6937, 7564, 8599, 8899]], ["sandal", [2425, 3341, 3762, 5198, 6295, 6315, 7159, 13851, 14083, 15082]]], "distant": [["pizza", [772, 1652, 4097, 4280, 5437, 5783, 6413, 6580, 7708, 8505]], ["volcano", [2102, 3625, 4186, 4376, 5475, 7381, 9827, 10640, 15386, 16018]], ["goldfish", [235, 261, 306, 993, 2240, 2847, 3149, 3734, 4654, 4676]], ["valley", [312, 1828, 2505, 2804, 4258, 5741, 6569, 7657, 8317, 9595]], ["traffic light", [273, 2674, 3208, 4645, 5151, 6592, 8100, 8816, 11587, 12798]]]}
THRESH = 90.375      # 30% of 5a:719's strongest peak over all 50,000 val photos
LECTURE_HITS10 = {"predicted": 22, "near_miss": 6, "residue": 13, "distant": 0}   # the lecture's scan, on these same photos
rates = {}
for g, classes in GROUPS.items():
    ids = [i for _, cl in classes for i in cl]
    p, _ = peaks(ids)
    rates[g] = (p >= THRESH).mean()
    print(f"{g:10s} {int((p >= THRESH).sum()):3d} hits of {len(ids)} photos ({rates[g]:.0%})")
    check(f"  {g}: hits", int((p >= THRESH).sum()), LECTURE_HITS10[g], tol=2)
check("the residue beats the predicted classes", bool(rates["residue"] > rates["predicted"]), True)

### Your turn 1 · Which predicted classes carry it?

Per class, which of the twelve predicted classes hit most? The lecture's 50-photo rates: monarch 66%, brassiere 52%,
bow tie 34%, … dumbbell 2%.

*Hint: loop over `GROUPS["predicted"]` and compute each class's hit share.*

In [ ]:
# TODO: hit share for each predicted class

## Plate VII · Steps 6–7: wiring in

The pool branch is the simplest wiring in the network. 5a's input (layer 4e's output, already halved to 7 × 7) goes
through a 3 × 3 **max-pool** (each position takes the largest value around it), then one weight per input channel. So
5a:719 = Σ wᵢ × maxpool(4e)ᵢ + b, with 832 weights. We build it by hand and compare.

In [ ]:
import torch.nn.functional as F
conv = net.mixed5a_pool_reduce_pre_relu_conv
grab = {}
h = net.mixed5a_1x1_pre_relu_conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0]))   # 5a's input
with torch.no_grad():
    net(next(batches([photo(S[top[0]])])))
h.remove()
x = grab["x"]                                           # 1 x 832 x 7 x 7
pooled = F.max_pool2d(x, 3, stride=1, padding=1)        # each position: the largest value in its 3 x 3 neighbourhood
w = conv.weight[15, :, 0, 0]; b = conv.bias[15]
by_hand = torch.einsum("bchw,c->bhw", pooled, w)[0] + b
network = taps[L5][0, N]
check("largest difference, by hand vs the network", float((by_hand - network).abs().max()), 0, tol=0.05)
W = w.cpu().numpy()
order = np.argsort(-W)
print("strongest inputs (4e channel, weight):", [(int(c), round(float(W[c]), 3)) for c in order[:5]])
for c, wv in [[163, 0.13935], [81, 0.11679], [616, 0.11488]]:
    check(f"  input 4e:{c}'s weight", W[c], wv, tol=1e-4)

Its strongest inputs are 4e:163 (school buses, tench, airships), 4e:81 (spiders, bow ties, monarchs) and 4e:616 (pickups,
golf carts). No single input is "pinched centre": like every neuron in the course, 5a:719 is a weighted vote over the
layer before it.

### Your turn 2 · Its strongest *negative* input

Which 4e channel pushes 5a:719 down hardest, and by how much? (The lecture: 4e:184, whose top photos are stupas and screws.)

*Hint: `np.argsort(W)`.*

In [ ]:
# TODO: 5a:719's most negative input weight

## Plate VIII · Step 9: ask it to draw

A free drawing of 5a:719, with the course's recipe (Lecture 8): random frequencies, decorrelated colour, small random
jitters each step, 512 steps. About 30 seconds on a T4.

In [ ]:
COLOR = np.array([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]], dtype=np.float32)
COLOR = COLOR / np.max(np.linalg.norm(COLOR, axis=0))


def fft_image(B, h=224, w=224, sd=0.01, seed=0):
    g = torch.Generator().manual_seed(seed)
    fy, fx = np.fft.fftfreq(h)[:, None], np.fft.fftfreq(w)[: w // 2 + 1]
    freqs = np.sqrt(fx * fx + fy * fy)
    spec = (torch.randn((B, 3) + freqs.shape + (2,), generator=g) * sd).to(DEV).requires_grad_(True)
    scale = torch.tensor(1.0 / np.maximum(freqs, 1.0 / max(w, h))).float()[None, None, ..., None].to(DEV)
    C = torch.tensor(COLOR, device=DEV)
    def image():
        x = torch.fft.irfftn(torch.view_as_complex(scale * spec), s=(h, w), norm="ortho") / 4.0
        return torch.sigmoid(torch.einsum("bchw,dc->bdhw", x, C))
    return spec, image


def jiggle(x):
    B = x.shape[0]
    x = F.pad(x, (12, 12, 12, 12), value=0.5)
    def jitter(x, d):
        dx, dy = np.random.randint(0, d + 1, 2)
        return x[:, :, dy:x.shape[2] - (d - dy), dx:x.shape[3] - (d - dx)]
    x = jitter(x, 8)
    x = F.interpolate(x, scale_factor=float(np.random.choice([1 + (i - 5) / 50.0 for i in range(11)])), mode="bilinear", align_corners=False)
    a = math.radians(float(np.random.choice(list(range(-10, 11)) + 5 * [0])))
    th = torch.tensor([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0]], device=x.device).float()
    x = F.grid_sample(x, F.affine_grid(th[None].expand(B, 2, 3), list(x.shape), align_corners=False), padding_mode="border", align_corners=False)
    x = jitter(x, 4)
    if min(x.shape[-2:]) < 224:
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
    return x


def draw(sign=1, steps=512, seed=0):
    np.random.seed(seed); torch.manual_seed(seed)
    spec, image = fft_image(1, seed=seed)
    opt = torch.optim.Adam([spec], lr=0.05)
    for _ in range(steps):
        opt.zero_grad(); net(to_input(jiggle(image())))
        (-sign * taps[L5][0, N].mean()).backward(); opt.step()
    with torch.no_grad():
        x = image(); net(to_input(x))
    return (x[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8), float(taps[L5][0, N].max())


t0 = time.time()
drawing, dv = draw()
print(f"drawn in {time.time() - t0:.0f} s; 5a:719 on its own drawing: {dv:.0f} (its strongest photo of the slice: {mx.max():.0f})")
check("the drawing drives it harder than any photo of the slice", bool(dv > mx.max()), True)
plt.figure(figsize=(3.4, 3.4)); plt.imshow(drawing); plt.axis("off"); plt.title("5a:719, drawn freely", fontsize=10); plt.show()

The lecture's drawings show paired lobes and wing-like shapes around a narrow middle: the pinch is in there. But a
drawing follows the easiest road up (30a), so it has to be checked against photos and against a test we control.

## Plate IX · Step 10: a controlled test, written first

The lecture drew two dark ellipses side by side on grey, the gap between them set by g (negative g: they overlap into a
**waist**, a pinched centre), and compared with one ellipse of the same total area. If 5a:719 wants a pinch, the waist
should win. The drawing code is the lecture's.

In [ ]:
from PIL import ImageDraw


def pair(g, size=160, angle=0.0, kind="pair", res=224, ss=4):
    """Two dark ellipses whose centres are (1 + g) widths apart (g < 0 overlaps into a waist), or one of the same area."""
    R = res * ss; im = Image.new("RGB", (R, R), (128, 128, 128)); d = ImageDraw.Draw(im)
    fg = (20, 20, 20); s = size * ss; a = math.radians(angle); cx = cy = R / 2
    rx, ry = s * 0.20, s * 0.14
    if kind == "single":
        k = math.sqrt(2)
        if angle == 0:
            d.ellipse([cx - rx * k, cy - ry * k, cx + rx * k, cy + ry * k], fill=fg)
        else:
            d.ellipse([cx - ry * k, cy - rx * k, cx + ry * k, cy + rx * k], fill=fg)
    else:
        o = rx * (1 + g)
        for sg in (-1, 1):
            ox, oy = sg * o * math.cos(a), sg * o * math.sin(a)
            if angle == 0:
                d.ellipse([cx + ox - rx, cy + oy - ry, cx + ox + rx, cy + oy + ry], fill=fg)
            else:
                d.ellipse([cx + ox - ry, cy + oy - rx, cx + ox + ry, cy + oy + rx], fill=fg)
    return np.asarray(im.resize((res, res), Image.LANCZOS))


def score(imgs):
    out = []
    with torch.no_grad():
        for x in batches(imgs):
            net(x); out += taps[L5][:, N].flatten(1).amax(1).tolist()
    return np.array(out)


G = [round(v, 2) for v in np.linspace(-0.6, 1.2, 10)]
waist = score([pair(g) for g in G])
one = score([pair(0, kind="single")])[0]
best = G[int(np.argmax(waist))]
check("the gap that drives it most (negative = overlapping, a waist)", best, -0.2)
check("its value", waist.max(), 47.034, tol=1.0)
check("one ellipse of the same area", one, 7.394, tol=1.0)

In [ ]:
fig = plt.figure(figsize=(11, 3.4))
for k, g in enumerate(G):
    a = fig.add_subplot(2, 11, k + 1); a.imshow(pair(g)); a.axis("off"); a.set_title(f"g={g}", fontsize=7)
a = fig.add_subplot(2, 11, 11); a.imshow(pair(0, kind="single")); a.axis("off"); a.set_title("one", fontsize=7)
a = fig.add_subplot(2, 1, 2)
a.plot(G, waist, color=INK, marker="o", mfc=PAPER); a.axhline(one, color=INK2, ls="--", lw=1)
a.text(G[-1], one, "one ellipse ", ha="right", va="bottom", fontsize=9, color=INK2)
a.set_xlabel("gap g between the two ellipses (negative: they overlap into a waist)"); a.set_ylabel("5a:719 peak")
plt.tight_layout(); plt.show()

A slight overlap, a waist, drives it about six times harder than a single ellipse of the same area: the pinch is real.
But compare the numbers: about 47 for the best drawing, against more than 90 for most loafer and sandal photos and up to
301 for the strongest photo of all. A pinched centre is part of what 5a:719 wants, not the whole of it.

### Your turn 3 · Stand the pair up

The lecture also stacked the two ellipses vertically (`angle=90`). Does the waist still win when the pinch runs the
other way?

*Hint: `score([pair(g, angle=90) for g in G])`, and compare with the horizontal sweep.*

In [ ]:
# TODO: the vertical sweep

## Plate XI · Step 12: take it away

Now cut the neuron out. We zero 5a:719 (after its ReLU) and see which answers of the network lose probability. The lecture
did it on the 2,000 val photos where 5a:719 fires most strongly, and the classes that lost most were **wallet**, **purse**
and **buckle**: not bow ties or butterflies. Here we repeat it on exactly those photos of those three classes.

In [ ]:
HURT_IDS = {"wallet": [34875, 16255, 20495, 865, 21288, 31482, 46826, 22594, 49452, 47366, 25156, 10980, 46585, 37383, 42765, 49757, 3107, 11816, 8402, 20640, 7168, 11937, 45316, 36252, 19232, 46163, 28531], "purse": [45887, 39204, 6211, 10948, 12284, 1564, 18489, 25943, 34221, 36410, 28360, 18578, 11670, 49450, 4190, 14523, 48123, 24777, 30667, 2589, 11829, 30486, 42670, 47619, 19536, 23253, 13341, 26614, 13521, 4460, 39355, 13450, 41903, 2267, 42229, 13577], "buckle": [34478, 49837, 39162, 17115, 20406, 24273, 26656, 41354, 12851, 22317, 8957, 2547, 3887, 1076, 32038, 30416, 7919, 35864, 4323, 48586, 45753, 46990, 12853, 30629, 15511]}          # of the 2,000 photos where 5a:719 fires most, the ones of these three classes
LECTURE = {"wallet": -0.2346, "purse": -0.1934, "buckle": -0.1574}           # mean change in the true class's probability, per class
relu_out = net.mixed5a_pool_reduce                        # the pool branch's ReLU, channel 15 is 5a:719


def probs(ids, cut):
    out = []
    h = relu_out.register_forward_hook(lambda m, i, o: o.index_fill(1, torch.tensor([15], device=o.device), 0.0)) if cut else None
    with torch.no_grad():
        for x in batches([photo(i) for i in ids]):
            out.append(net(x).cpu().numpy())
    if h: h.remove()
    return np.concatenate(out)


for cls, ids in HURT_IDS.items():
    fetch(ids)
    k = LUCID.index(cls)
    d = (probs(ids, True)[:, k] - probs(ids, False)[:, k]).mean()
    print(f"{cls:8s} {len(ids):3d} photos: true-class probability changes by {minus(f'{d:+.3f}')}")
    check(f"  {cls}: mean change", d, LECTURE[cls], tol=0.02)

Taking 5a:719 away hurts wallets, purses and buckles most: leather things with a clasp or a fold at the middle. That is
the residue the essay could not explain, and it is where the neuron matters most to the network's answers.

## Plate XII · Steps 13–15: what is left

- **13 · Growth.** InceptionV1 (2014) has no training checkpoints, so we cannot watch 5a:719 form.
- **14 · Alternatives and residue.** The lecture weighed five causes for the "exceptions"; the photos say they are not
  exceptions at all: loafers and sandals fire 71% of the time, more than any predicted class.
- **15 · The explanation.** The lecture's sentence: *5a:719 responds to a dark, compact, two-lobed shape with a pinched
  middle, most strongly in leather goods (shoes, wallets, purses), less in bow ties and butterflies; scope: InceptionV1
  on ImageNet; confidence: moderate.* The winning description was written after seeing the top photos, so it is not
  blind: the lecture says so.

### Your turn 4 · The random control

The lecture repeated the ablation on 2,000 random photos as a control. Take 300 random photos of the slice, cut 5a:719,
and count how many change their top answer. (The lecture: 252 of the 2,000 strong photos flipped.)

*Hint: compare `probs(ids, True).argmax(1)` with `probs(ids, False).argmax(1)`.*

In [ ]:
# TODO: how many of 300 random slice photos flip their top answer

## Plate XIII · Your turn: the assignment

Choose one neuron (a list is in the lecture) and take it through the fifteen steps. Submit a JSON file in the format of
the lecture's template (`files/submission_template.json` on Plate XIII), with your one-sentence explanation, your
pre-registered classes and your measurements. The lecture's grader (`grader/grade.py`) re-measures your claims on
ImageNet-val and scores them against a 100-point rubric: the required steps, a test written before measuring, an
explanation that can lose, an ablation, and an honest residue.

### Your turn 5 · Start your own

Pick one neuron of 5a or 5b (try 5b:342, the lecture's messiest), and do Steps 1–3 on the 3,000-photo slice with
`peaks(S, "mixed5b", 342)`: how often does it fire, and what are its strongest classes?

In [ ]:
# TODO: Steps 1-3 for a neuron of your choice

## Solutions

**Your turn 1 · Which predicted classes carry it?**

In [ ]:
for name, ids in GROUPS["predicted"]:
    p, _ = peaks(ids)
    print(f"{name:18s} {(p >= THRESH).mean():.0%} of 10")

On ten photos each, bow ties and cabbage butterflies hit most, and dumbbells, ringlets and sulphur butterflies not at all. With all 50 (the lecture), monarchs (66%) and brassieres (52%) lead and bow ties reach 34%. Either way the pinched-centre classes are uneven, which is why the description loses.

**Your turn 2 · Its strongest negative input.**

In [ ]:
neg = np.argsort(W)[:3]
print([(int(c), round(float(W[c]), 3)) for c in neg])
check("its most negative input", int(neg[0]), 184)
check("that weight", W[neg[0]], -0.09073, tol=1e-4)

**Your turn 3 · Stand the pair up.**

In [ ]:
upright = score([pair(g, angle=90) for g in G])
print("vertical:", np.round(upright, 1))
check("the vertical sweep's strongest value", upright.max(), 15.296, tol=1.0)
check("the horizontal waist beats every vertical pair", bool(waist.max() > upright.max()), True)

Standing up, nothing reaches the horizontal waist: 5a:719 wants a pinch with lobes either side, like a bow tie or a pair of wings, not stacked.

**Your turn 4 · The random control.**

In [ ]:
rng = np.random.default_rng(0)
ids = list(rng.choice(S, 300, replace=False))
flips = int((probs(ids, True).argmax(1) != probs(ids, False).argmax(1)).sum())
print(f"{flips} of 300 random photos change their top answer when 5a:719 is cut")
check("fewer flip than among strong photos (lecture: 252 of 2,000 strong)", flips / 300 < 252 / 2000, True)

Few: on photos where it barely fires, cutting 5a:719 changes little. Its effect is concentrated where it fires.

**Your turn 5 · Start your own.**

In [ ]:
m342, n342 = peaks(S, "mixed5b", 342)
print(f"5b:342 fires on {(m342 > 0).mean():.0%} of the slice")
print("its strongest classes:", Counter(NAMES[c] for c in labels[np.argsort(-m342)[:30]]).most_common(6))

Then Steps 4–15, on your own. The lecture's worked submission for 5a:719 (`files/w719_submission.json`) shows what a complete one looks like.